# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuan05-afk/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

Lane locked: Refresh / Content Opportunity Scoring.


## 1. My lane as an ML task (type)

**Primary task type:** ranking / scoring (produce a priority score, sort top-K for review).

**Secondary view:** binary classification of `is_declining_label` to train and evaluate the
score, with Precision@K as the operational metric editors care about.

This is not clustering and not pure CTR scoring; the decision is "which page first?".


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path('.').resolve()
# Support running from repo root or work/notebooks
if (ROOT / 'data/raw/content_refresh_anonymized.csv').exists():
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
elif (ROOT.parents[1] / 'data/raw/content_refresh_anonymized.csv').exists():
    ROOT = ROOT.parents[1]
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
else:
    # Colab / nested
    for cand in [ROOT, *ROOT.parents]:
        if (cand / 'data/raw/content_refresh_anonymized.csv').exists():
            ROOT = cand
            DATA = cand / 'data/raw/content_refresh_anonymized.csv'
            break
    else:
        raise FileNotFoundError('data/raw/content_refresh_anonymized.csv')

WORK_OUT = ROOT / 'work' / 'outputs'
WORK_OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
print(f'Loaded {len(df):,} rows × {df.shape[1]} cols from {DATA.relative_to(ROOT)}')
print(f'Clients (pseudonymous): {df["client_id"].nunique()}')


Loaded 30,000 rows × 44 cols from data\raw\content_refresh_anonymized.csv
Clients (pseudonymous): 32


## 2. Target or proxy

**Target:** `is_declining_label` — derived in the pipeline from observed `trend_direction`
(down vs not). It is an observed proxy for "page shows decline in the measurement window,"
not a hand-labeled editorial truth.

**Leakage rule:** `trend_direction` and `trend_pct` are NEVER features (they define the label).


In [2]:
# Show how the label relates to trend_direction (label is observed-proxy, not editorial judgment)
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
ct = pd.crosstab(df['trend_direction'], df['is_declining_label'])
print(ct)
print('positive rate:', round(df['is_declining_label'].mean(), 3))


is_declining_label     0      1
trend_direction                
down                   0  16262
flat                1152      0
new                 2236      0
stable              5962      0
up                  4388      0
positive rate: 0.542


## 3. Success metric

**Primary:** Precision@50 on a **client-holdout** test set (share of top-50 scored pages that
are declining). Editors review a short list; precision at that list length is the honest bar.

**Secondary:** ROC-AUC and average precision for discrimination; always report next to the
**base rate** (~54% declining) so a high precision is not confused with a high base rate.

**Baseline to beat:** the transparent rule score from `scripts/02_baseline_score.py`.


In [3]:
base = float(df['is_declining_label'].mean()) if 'is_declining_label' in df.columns else float((df['trend_direction']=='down').mean())
print(f'Base rate (declining): {base:.3f}')
print('Success = Precision@50 > baseline Precision@50 on the SAME client-holdout split.')
print('Reference pipeline (run_all): RF Precision@50 ≈ 0.74 vs baseline ≈ 0.24 on that split.')


Base rate (declining): 0.542
Success = Precision@50 > baseline Precision@50 on the SAME client-holdout split.
Reference pipeline (run_all): RF Precision@50 ≈ 0.74 vs baseline ≈ 0.24 on that split.


## 4. The unit of analysis, as a real dataframe

One row = one pseudonymized **content item** (page), with trailing-90-day search/analytics
aggregates. IDs are for grouping/splits only.


In [4]:
unit = df[['content_id','client_id','impressions_90d','clicks_90d','avg_position','ctr',
           'days_since_last_update','trend_direction','content_type']].head(8)
print('One row = one content item. Preview (no titles/URLs):')
display(unit) if 'display' in dir() else print(unit.to_string())
print(f'\nShape: {df.shape} | unique content_id: {df["content_id"].nunique()}')


One row = one content item. Preview (no titles/URLs):
             content_id          client_id  impressions_90d  clicks_90d  avg_position   ctr  days_since_last_update trend_direction     content_type
0  content_304f48230142  client_f369cb89fc             3803          29          10.6  0.76                      20            down  keyword article
1  content_a1fb4e703a9e  client_4e07408562            15320           7          20.3  0.05                      25            down  keyword article
2  content_9aa793d4d895  client_7f2253d7e2            12581          11          36.5  0.09                      20            down  keyword article
3  content_331d6c4de07b  client_19581e27de            11751          58           6.2  0.49                      22          stable  keyword article
4  content_d99b7a2d90ca  client_3fdba35f04            19140          24          44.0  0.13                      14            down  keyword article
5  content_d4084a4bc775  client_f369cb89fc          

## 5. Why ML beats a fixed rule here

A fixed rule (e.g. "if stale and impressions high → refresh") is a fair baseline and stays
in the playbook as reason codes. It fails when:

1. Multiple failure modes compete (CTR gap vs thin content vs staleness).
2. Thresholds need to trade precision vs coverage across clients with different volumes.
3. Interactions (position × CTR × impressions) are messy to encode by hand.

ML earns a place only if it beats that transparent baseline on Precision@K under client-holdout.
If it does not, the baseline rule remains the production recommendation.


In [5]:
# Illustrate multi-signal messiness: decline rate by crude buckets
tmp = df.copy()
tmp['stale'] = tmp['days_since_last_update'] >= 180
tmp['visible'] = tmp['impressions_90d'] >= 500
tmp['label'] = (tmp['trend_direction']=='down').astype(int)
g = tmp.groupby(['stale','visible'])['label'].agg(['mean','count'])
print(g)
print('\nDecline rates differ across buckets — one if-statement oversimplifies.')


                   mean  count
stale visible                 
False False    0.475337  13117
      True     0.595188  16709
True  False    0.420382    157
      True     0.941176     17

Decline rates differ across buckets — one if-statement oversimplifies.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
